# Object Detection with YOLOv3 and OpenCV

We will use YOLOv3 (You Only Look Once) to perform object detection on a video stream. We will use OpenCV to capture video from a camera and display the detection results. 

## Setup

First, let's import the necessary libraries and load the YOLOv3 model configuration and weights.

### Import Libraries:

- `cv2`: OpenCV library for computer vision tasks.
- `numpy as np`: Fundamental package for numerical computing in Python.

### Load YOLOv3 Model:

- Load the YOLOv3 model architecture from the yolov3.cfg file and the pretrained weights from `yolov3.weights`.

### Load Class Names:

- Open the file coco.names in read mode.
- Read class names into a list.

### Get Output Layer Names:

- Retrieve all layer names from the network.
- Get the names of the output layers


In [1]:
import cv2
import numpy as np

# Load YOLOv3 configuration and weights files
net = cv2.dnn.readNet('yolov3.weights', 'yolov3.cfg')

# Load class names
with open('coco.names', 'r') as f:
    classes = f.read().strip().split('\n')

# Get the output layer names
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

## Video Capture and Object Detection

Next, we will capture video from the camera and perform object detection on each frame.

### 1. Video Capture Initialization:

- `cap = cv2.VideoCapture(0)`: Initialize video capture from the default camera.

### 2. Frame Processing:

- Capture Frame: `ret, frame = cap.read()`: Read a frame from the video capture.
- Get Dimensions: `height, width = frame.shape[:2]`: Get the frame's dimensions.
- Create Blob: `blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)`: Convert the frame to a blob for the network.
- Set Input: `net.setInput(blob)`: Set the blob as input to the network.
- Forward Pass: `outputs = net.forward(output_layers)`: Perform a forward pass to get detections.

### 3. Detection Processing:

- Initialize Lists: Initialize lists for bounding boxes, confidences, and class IDs.
- Iterate Detections: Iterate over detections and extract bounding box coordinates, confidences, and class IDs for detections with `confidence > 0.5`.
- Non-Maximum Suppression: Apply Non-Maximum Suppression to filter out redundant bounding boxes.
- Draw Bounding Boxes: Draw bounding boxes and labels on the frame.

### 4. Display and Terminate:

- Display Frame: `cv2.imshow('YOLO Object Detection', frame)`: Display the frame with detections.
- Exit Loop: Break the loop if '`q`' key is pressed.
- Release Resources: Release the video capture object and close all OpenCV windows.

In [5]:
import cv2
import numpy as np

# Load YOLOv3 configuration and weights files
net = cv2.dnn.readNet('yolov3.weights', 'yolov3.cfg')

# Load class names
with open('coco.names', 'r') as f:
    classes = f.read().strip().split('\n')

# Get the output layer names
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

# Initialize the video capture object for the camera (use 0 for the default camera)
cap = cv2.VideoCapture('videos/albania.mp4')
"""
    Use videos
    videos/albania.mp4
    videos/crowd.mp4
    videos/shoppingmall.mp4
"""

while True:
    # Read a frame from the video capture
    ret, frame = cap.read()
    if not ret:
        break
    
    # Get the frame dimensions
    height, width = frame.shape[:2]

    # Create a blob from the frame
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)

    # Perform forward pass
    outputs = net.forward(output_layers)

    # Initialize lists for detected bounding boxes, confidences, and class IDs
    boxes = []
    confidences = []
    class_ids = []

    # Iterate over each detection
    for output in outputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > 0.5:
                # Scale bounding box coordinates to the original frame size
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)

                # Calculate the top-left corner of the bounding box
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                # Append to lists
                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Apply Non-Maximum Suppression
    indices = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

    # Draw bounding boxes and labels on the frame
    if len(indices) > 0:
        for i in indices.flatten():
            box = boxes[i]
            x, y, w, h = box
            label = str(classes[class_ids[i]])
            confidence = confidences[i]

            # Draw the bounding box
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Display the label and confidence score
            text = f"{label}: {confidence:.2f}"
            cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display the frame with detections
    cv2.imshow('YOLO Object Detection', frame)

    # Break the loop if 'q' key is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture object and close all OpenCV windows
cap.release()
cv2.destroyAllWindows()


### Happy Coding